In [0]:
env     = "dev"
regions = ["africa", "america", "asia", "europe", "middle_east"]
BR, SV, GD = f"{env}_bronze", f"{env}_silver", f"{env}_gold"
AUDIT = """,
  _ingested_at TIMESTAMP COMMENT 'When the row was loaded',
  _source      STRING    COMMENT 'Source table the row came from'"""
def create_table(fq_name, columns, comment, audit=False):
    cols = columns + (AUDIT if audit else "")
    spark.sql(f"CREATE TABLE IF NOT EXISTS {fq_name} ({cols}) COMMENT '{comment}'")
    print("✔", fq_name)


In [0]:
bronze_common = {
  "region":   ("r_regionkey BIGINT, r_name STRING, r_comment STRING", "Raw TPC-H region"),
  "nation":   ("n_nationkey BIGINT, n_name STRING, n_regionkey BIGINT, n_comment STRING", "Raw TPC-H nation"),
  "part":     ("""p_partkey BIGINT, p_name STRING, p_mfgr STRING, p_brand STRING, p_type STRING,
                 p_size INT, p_container STRING, p_retailprice DECIMAL(18,2), p_comment STRING""", "Raw TPC-H part"),
  "supplier": ("""s_suppkey BIGINT, s_name STRING, s_address STRING, s_nationkey BIGINT,
                 s_phone STRING, s_acctbal DECIMAL(18,2), s_comment STRING""", "Raw TPC-H supplier"),
  "partsupp": ("""ps_partkey BIGINT, ps_suppkey BIGINT, ps_availqty INT,
                 ps_supplycost DECIMAL(18,2), ps_comment STRING""", "Raw TPC-H partsupp"),
}
bronze_regional = {
  "customer": ("""c_custkey BIGINT, c_name STRING, c_address STRING, c_nationkey BIGINT, c_phone STRING,
                 c_acctbal DECIMAL(18,2), c_mktsegment STRING, c_comment STRING""", "Raw TPC-H customer"),
  "orders":   ("""o_orderkey BIGINT, o_custkey BIGINT, o_orderstatus STRING, o_totalprice DECIMAL(18,2),
                 o_orderdate DATE, o_orderpriority STRING, o_clerk STRING, o_shippriority INT, o_comment STRING""",
               "Raw TPC-H orders"),
  "lineitem": ("""l_orderkey BIGINT, l_partkey BIGINT, l_suppkey BIGINT, l_linenumber INT,
                 l_quantity DECIMAL(18,2), l_extendedprice DECIMAL(18,2), l_discount DECIMAL(18,2),
                 l_tax DECIMAL(18,2), l_returnflag STRING, l_linestatus STRING, l_shipdate DATE,
                 l_commitdate DATE, l_receiptdate DATE, l_shipinstruct STRING, l_shipmode STRING,
                 l_comment STRING""", "Raw TPC-H lineitem"),
}
for t, (cols, cmt) in bronze_common.items():
    create_table(f"{BR}.common.{t}", cols, cmt, audit=True)
for r in regions:
    for t, (cols, cmt) in bronze_regional.items():
        create_table(f"{BR}.{r}.{t}", cols, f"{cmt} - {r} sales region", audit=True)


In [0]:
silver_common = {
  "regions": ("""
    region_id   BIGINT NOT NULL COMMENT 'Region ID',
    region_name STRING          COMMENT 'Region name',
    CONSTRAINT pk_regions PRIMARY KEY (region_id)""", "Cleaned regions"),
  "nations": ("""
    nation_id   BIGINT NOT NULL COMMENT 'Nation ID',
    nation_name STRING          COMMENT 'Country name',
    region_id   BIGINT          COMMENT 'Region ID',
    CONSTRAINT pk_nations PRIMARY KEY (nation_id)""", "Cleaned nations"),
  "products": ("""
    product_id   BIGINT NOT NULL COMMENT 'Product ID',
    product_name STRING, manufacturer STRING, brand STRING, product_type STRING,
    size INT, container STRING, retail_price DECIMAL(18,2),
    CONSTRAINT pk_products PRIMARY KEY (product_id)""", "Cleaned products (from part)"),
  "suppliers": ("""
    supplier_id     BIGINT NOT NULL COMMENT 'Supplier ID',
    supplier_name   STRING, address STRING, nation_id BIGINT, phone STRING,
    account_balance DECIMAL(18,2),
    CONSTRAINT pk_suppliers PRIMARY KEY (supplier_id)""", "Cleaned suppliers"),
  "product_suppliers": ("""
    product_id    BIGINT NOT NULL, supplier_id BIGINT NOT NULL,
    available_qty INT, supply_cost DECIMAL(18,2),
    CONSTRAINT pk_product_suppliers PRIMARY KEY (product_id, supplier_id)""",
    "Which supplier supplies which product, with cost"),
}
silver_regional = {
  "customers": ("""
    customer_id     BIGINT NOT NULL COMMENT 'Customer ID',
    customer_name   STRING, address STRING, nation_id BIGINT, phone STRING,
    account_balance DECIMAL(18,2), market_segment STRING,
    CONSTRAINT pk_customers PRIMARY KEY (customer_id)""", "Cleaned customers"),
  "orders": ("""
    order_id          BIGINT NOT NULL COMMENT 'Order ID',
    customer_id       BIGINT,
    order_status_code STRING COMMENT 'F, O or P',
    order_status      STRING COMMENT 'Fulfilled, Open, Partially fulfilled',
    total_price       DECIMAL(18,2), order_date DATE, order_priority STRING,
    clerk STRING, ship_priority INT,
    CONSTRAINT pk_orders PRIMARY KEY (order_id)""", "Cleaned orders"),
  "order_lines": ("""
    order_id BIGINT NOT NULL, line_number INT NOT NULL,
    product_id BIGINT, supplier_id BIGINT,
    quantity DECIMAL(18,2), extended_price DECIMAL(18,2),
    discount_pct DECIMAL(18,2), tax_pct DECIMAL(18,2),
    return_flag STRING, line_status STRING,
    ship_date DATE, commit_date DATE, receipt_date DATE,
    ship_instruction STRING, ship_mode STRING,
    CONSTRAINT pk_order_lines PRIMARY KEY (order_id, line_number)""", "Cleaned order lines"),
}
for t, (cols, cmt) in silver_common.items():
    create_table(f"{SV}.common.{t}", cols, cmt)
for r in regions:
    for t, (cols, cmt) in silver_regional.items():
        create_table(f"{SV}.{r}.{t}", cols, f"{cmt} - {r} sales region")


In [0]:
silver_common = {
  "regions": ("""
    region_id   BIGINT NOT NULL COMMENT 'Region ID',
    region_name STRING          COMMENT 'Region name',
    CONSTRAINT pk_regions PRIMARY KEY (region_id)""", "Cleaned regions"),
  "nations": ("""
    nation_id   BIGINT NOT NULL COMMENT 'Nation ID',
    nation_name STRING          COMMENT 'Country name',
    region_id   BIGINT          COMMENT 'Region ID',
    CONSTRAINT pk_nations PRIMARY KEY (nation_id)""", "Cleaned nations"),
  "products": ("""
    product_id   BIGINT NOT NULL COMMENT 'Product ID',
    product_name STRING, manufacturer STRING, brand STRING, product_type STRING,
    size INT, container STRING, retail_price DECIMAL(18,2),
    CONSTRAINT pk_products PRIMARY KEY (product_id)""", "Cleaned products (from part)"),
  "suppliers": ("""
    supplier_id     BIGINT NOT NULL COMMENT 'Supplier ID',
    supplier_name   STRING, address STRING, nation_id BIGINT, phone STRING,
    account_balance DECIMAL(18,2),
    CONSTRAINT pk_suppliers PRIMARY KEY (supplier_id)""", "Cleaned suppliers"),
  "product_suppliers": ("""
    product_id    BIGINT NOT NULL, supplier_id BIGINT NOT NULL,
    available_qty INT, supply_cost DECIMAL(18,2),
    CONSTRAINT pk_product_suppliers PRIMARY KEY (product_id, supplier_id)""",
    "Which supplier supplies which product, with cost"),
}
silver_regional = {
  "customers": ("""
    customer_id     BIGINT NOT NULL COMMENT 'Customer ID',
    customer_name   STRING, address STRING, nation_id BIGINT, phone STRING,
    account_balance DECIMAL(18,2), market_segment STRING,
    CONSTRAINT pk_customers PRIMARY KEY (customer_id)""", "Cleaned customers"),
  "orders": ("""
    order_id          BIGINT NOT NULL COMMENT 'Order ID',
    customer_id       BIGINT,
    order_status_code STRING COMMENT 'F, O or P',
    order_status      STRING COMMENT 'Fulfilled, Open, Partially fulfilled',
    total_price       DECIMAL(18,2), order_date DATE, order_priority STRING,
    clerk STRING, ship_priority INT,
    CONSTRAINT pk_orders PRIMARY KEY (order_id)""", "Cleaned orders"),
  "order_lines": ("""
    order_id BIGINT NOT NULL, line_number INT NOT NULL,
    product_id BIGINT, supplier_id BIGINT,
    quantity DECIMAL(18,2), extended_price DECIMAL(18,2),
    discount_pct DECIMAL(18,2), tax_pct DECIMAL(18,2),
    return_flag STRING, line_status STRING,
    ship_date DATE, commit_date DATE, receipt_date DATE,
    ship_instruction STRING, ship_mode STRING,
    CONSTRAINT pk_order_lines PRIMARY KEY (order_id, line_number)""", "Cleaned order lines"),
}
for t, (cols, cmt) in silver_common.items():
    create_table(f"{SV}.common.{t}", cols, cmt)
for r in regions:
    for t, (cols, cmt) in silver_regional.items():
        create_table(f"{SV}.{r}.{t}", cols, f"{cmt} - {r} sales region")


In [0]:
G = f"{GD}.common"   # shared dimensions
# --- Shared dimensions (create first, because facts reference them) ---
create_table(f"{G}.dim_date", """
  date_key     INT NOT NULL COMMENT 'Date key in YYYYMMDD format, e.g. 19960315',
  full_date    DATE    COMMENT 'Calendar date',
  day_of_week  INT     COMMENT '1 = Sunday ... 7 = Saturday',
  day_name     STRING  COMMENT 'Monday, Tuesday, ...',
  month_num    INT     COMMENT 'Month number 1-12',
  month_name   STRING  COMMENT 'January, February, ...',
  quarter      INT     COMMENT 'Calendar quarter 1-4',
  year         INT     COMMENT 'Calendar year',
  is_weekend   BOOLEAN COMMENT 'True for Saturday and Sunday',
  CONSTRAINT pk_dim_date PRIMARY KEY (date_key)""",
  "Calendar dimension. Join on date_key for any time-based analysis.")
create_table(f"{G}.dim_geography", """
  nation_id   BIGINT NOT NULL COMMENT 'Unique country ID',
  nation_name STRING COMMENT 'Country name in upper case, e.g. GERMANY, BRAZIL',
  region_id   BIGINT COMMENT 'World region ID',
  region_name STRING COMMENT 'World region: AFRICA, AMERICA, ASIA, EUROPE, MIDDLE EAST',
  CONSTRAINT pk_dim_geography PRIMARY KEY (nation_id)""",
  "Geography dimension: country and world region.")
create_table(f"{G}.dim_product", """
  product_id   BIGINT NOT NULL COMMENT 'Unique product ID',
  product_name STRING COMMENT 'Product description',
  manufacturer STRING COMMENT 'Manufacturer, e.g. Manufacturer#1',
  brand        STRING COMMENT 'Brand, e.g. Brand#13',
  product_type STRING COMMENT 'Product type, e.g. PROMO BRUSHED COPPER',
  size         INT    COMMENT 'Product size',
  container    STRING COMMENT 'Packaging, e.g. SM BOX, LG CASE',
  retail_price DECIMAL(18,2) COMMENT 'List price per unit in USD',
  CONSTRAINT pk_dim_product PRIMARY KEY (product_id)""",
  "Product dimension: what is sold.")
create_table(f"{G}.dim_supplier", """
  supplier_id     BIGINT NOT NULL COMMENT 'Unique supplier ID',
  supplier_name   STRING COMMENT 'Supplier name, e.g. Supplier#000000001',
  nation_id       BIGINT COMMENT 'Supplier country; joins to dim_geography',
  phone           STRING COMMENT 'Supplier phone number',
  account_balance DECIMAL(18,2) COMMENT 'Supplier account balance in USD',
  CONSTRAINT pk_dim_supplier PRIMARY KEY (supplier_id),
  CONSTRAINT fk_supplier_geo FOREIGN KEY (nation_id) REFERENCES {G}.dim_geography(nation_id)""".format(G=G),
  "Supplier dimension: who supplies the products.")
# --- Regional dimension and facts ---
for r in regions:
    S = f"{GD}.{r}"
        print(S)
        create_table(f"{S}.dim_customer", f"""
        customer_id     BIGINT NOT NULL COMMENT 'Unique customer ID',
        customer_name   STRING COMMENT 'Customer name, e.g. Customer#000000001',
        market_segment  STRING COMMENT 'AUTOMOBILE, BUILDING, FURNITURE, HOUSEHOLD or MACHINERY',
        nation_id       BIGINT COMMENT 'Customer country; joins to common.dim_geography',
        phone           STRING COMMENT 'Customer phone number',
        account_balance DECIMAL(18,2) COMMENT 'Customer account balance in USD',
        CONSTRAINT pk_dim_customer PRIMARY KEY (customer_id),
        CONSTRAINT fk_customer_geo FOREIGN KEY (nation_id) REFERENCES {G}.dim_geography(nation_id)""",
        f"Customer dimension for the {r} sales region.")
        create_table(f"{S}.fact_orders", f"""
        order_id       BIGINT NOT NULL COMMENT 'Unique order ID',
        customer_id    BIGINT COMMENT 'Customer who placed the order',
        order_date_key INT    COMMENT 'Order date as YYYYMMDD; joins to common.dim_date',
        order_date     DATE   COMMENT 'Date the order was placed',
        order_status   STRING COMMENT 'Fulfilled, Open or Partially fulfilled',
        order_priority STRING COMMENT '1-URGENT, 2-HIGH, 3-MEDIUM, 4-NOT SPECIFIED, 5-LOW',
        clerk          STRING COMMENT 'Clerk who handled the order',
        total_amount   DECIMAL(18,2) COMMENT 'Order total in USD (revenue for the order)',
        line_count     INT    COMMENT 'Number of line items in the order',
        CONSTRAINT pk_fact_orders PRIMARY KEY (order_id),
        CONSTRAINT fk_orders_customer FOREIGN KEY (customer_id) REFERENCES {S}.dim_customer(customer_id),
        CONSTRAINT fk_orders_date FOREIGN KEY (order_date_key) REFERENCES {G}.dim_date(date_key)""",
        f"Order header fact, one row per order, {r} sales region.")
        create_table(f"{S}.fact_order_lines", f"""
        order_id      BIGINT NOT NULL COMMENT 'Order ID; joins to fact_orders',
        line_number   INT    NOT NULL COMMENT 'Line number within the order',
        product_id    BIGINT COMMENT 'Product sold; joins to common.dim_product',
        supplier_id   BIGINT COMMENT 'Supplier of the product; joins to common.dim_supplier',
        quantity      DECIMAL(18,2) COMMENT 'Units sold',
        gross_amount  DECIMAL(18,2) COMMENT 'Quantity x price before discount, USD',
        discount_pct  DECIMAL(18,2) COMMENT 'Discount as a fraction, e.g. 0.05 = 5 percent',
        net_amount    DECIMAL(18,2) COMMENT 'Revenue after discount: gross_amount x (1 - discount_pct), USD',
        tax_pct       DECIMAL(18,2) COMMENT 'Tax rate as a fraction',
        return_flag   STRING COMMENT 'R = returned, A = accepted, N = not yet decided',
        line_status   STRING COMMENT 'O = open, F = fulfilled',
        ship_date     DATE   COMMENT 'Date the line was shipped',
        ship_date_key INT    COMMENT 'Ship date as YYYYMMDD; joins to common.dim_date',
        ship_mode     STRING COMMENT 'AIR, RAIL, SHIP, TRUCK, MAIL, FOB or REG AIR',
        CONSTRAINT pk_fact_order_lines PRIMARY KEY (order_id, line_number),
        CONSTRAINT fk_lines_order    FOREIGN KEY (order_id)      REFERENCES {S}.fact_orders(order_id),
        CONSTRAINT fk_lines_product  FOREIGN KEY (product_id)    REFERENCES {G}.dim_product(product_id),
        CONSTRAINT fk_lines_supplier FOREIGN KEY (supplier_id)   REFERENCES {G}.dim_supplier(supplier_id),
        CONSTRAINT fk_lines_shipdate FOREIGN KEY (ship_date_key) REFERENCES {G}.dim_date(date_key)""",
        f"Order line fact, one row per product line in an order, {r} sales region. Main table for revenue and quantity.")
        # --- Combined views across all regions (used for text-to-SQL) ---



In [0]:

env     = "dev"
regions = ["africa", "america", "asia", "europe", "middle_east"] 

for r in regions:
    S = f"{GD}.{r}"
    print(S)
    create_table(f"{S}.dim_customer", f"""
        customer_id     BIGINT NOT NULL COMMENT 'Unique customer ID',
        customer_name   STRING COMMENT 'Customer name, e.g. Customer#000000001',
        market_segment  STRING COMMENT 'AUTOMOBILE, BUILDING, FURNITURE, HOUSEHOLD or MACHINERY',
        nation_id       BIGINT COMMENT 'Customer country; joins to common.dim_geography',
        phone           STRING COMMENT 'Customer phone number',
        account_balance DECIMAL(18,2) COMMENT 'Customer account balance in USD',
        CONSTRAINT pk_dim_customer PRIMARY KEY (customer_id),
        CONSTRAINT fk_customer_geo FOREIGN KEY (nation_id) REFERENCES {G}.dim_geography(nation_id)""",
        f"Customer dimension for the {r} sales region.")

In [0]:
for r in regions:
    S = f"{GD}.{r}"
    print(S)
    create_table(f"{S}.fact_orders", f"""
        order_id       BIGINT NOT NULL COMMENT 'Unique order ID',
        customer_id    BIGINT COMMENT 'Customer who placed the order',
        order_date_key INT    COMMENT 'Order date as YYYYMMDD; joins to common.dim_date',
        order_date     DATE   COMMENT 'Date the order was placed',
        order_status   STRING COMMENT 'Fulfilled, Open or Partially fulfilled',
        order_priority STRING COMMENT '1-URGENT, 2-HIGH, 3-MEDIUM, 4-NOT SPECIFIED, 5-LOW',
        clerk          STRING COMMENT 'Clerk who handled the order',
        total_amount   DECIMAL(18,2) COMMENT 'Order total in USD (revenue for the order)',
        line_count     INT    COMMENT 'Number of line items in the order',
        CONSTRAINT pk_fact_orders PRIMARY KEY (order_id),
        CONSTRAINT fk_orders_customer FOREIGN KEY (customer_id) REFERENCES {S}.dim_customer(customer_id),
        CONSTRAINT fk_orders_date FOREIGN KEY (order_date_key) REFERENCES {G}.dim_date(date_key)""",
        f"Order header fact, one row per order, {r} sales region.")
    create_table(f"{S}.fact_order_lines", f"""
        order_id      BIGINT NOT NULL COMMENT 'Order ID; joins to fact_orders',
        line_number   INT    NOT NULL COMMENT 'Line number within the order',
        product_id    BIGINT COMMENT 'Product sold; joins to common.dim_product',
        supplier_id   BIGINT COMMENT 'Supplier of the product; joins to common.dim_supplier',
        quantity      DECIMAL(18,2) COMMENT 'Units sold',
        gross_amount  DECIMAL(18,2) COMMENT 'Quantity x price before discount, USD',
        discount_pct  DECIMAL(18,2) COMMENT 'Discount as a fraction, e.g. 0.05 = 5 percent',
        net_amount    DECIMAL(18,2) COMMENT 'Revenue after discount: gross_amount x (1 - discount_pct), USD',
        tax_pct       DECIMAL(18,2) COMMENT 'Tax rate as a fraction',
        return_flag   STRING COMMENT 'R = returned, A = accepted, N = not yet decided',
        line_status   STRING COMMENT 'O = open, F = fulfilled',
        ship_date     DATE   COMMENT 'Date the line was shipped',
        ship_date_key INT    COMMENT 'Ship date as YYYYMMDD; joins to common.dim_date',
        ship_mode     STRING COMMENT 'AIR, RAIL, SHIP, TRUCK, MAIL, FOB or REG AIR',
        CONSTRAINT pk_fact_order_lines PRIMARY KEY (order_id, line_number),
        CONSTRAINT fk_lines_order    FOREIGN KEY (order_id)      REFERENCES {S}.fact_orders(order_id),
        CONSTRAINT fk_lines_product  FOREIGN KEY (product_id)    REFERENCES {G}.dim_product(product_id),
        CONSTRAINT fk_lines_supplier FOREIGN KEY (supplier_id)   REFERENCES {G}.dim_supplier(supplier_id),
        CONSTRAINT fk_lines_shipdate FOREIGN KEY (ship_date_key) REFERENCES {G}.dim_date(date_key)""",
        f"Order line fact, one row per product line in an order, {r} sales region. Main table for revenue and quantity.")

In [0]:
views = {
        "vw_customers_all":   ("dim_customer",     "All customers from every sales region"),
        "vw_orders_all":      ("fact_orders",      "All orders from every sales region"),
        "vw_order_lines_all": ("fact_order_lines", "All order lines from every sales region"),
        }
for v, (tbl, cmt) in views.items():
            union_sql = "\nUNION ALL\n".join(
                f"SELECT '{r.replace('_', ' ').upper()}' AS sales_region, * FROM {GD}.{r}.{tbl}" for r in regions)
            spark.sql(f"CREATE OR REPLACE VIEW {G}.{v} COMMENT '{cmt}. sales_region = region of the customer.' AS {union_sql}")
            print("✔ view", f"{G}.{v}")